# Milestone 2 — GenAI Prompt Engineering
## Real-Time Retail Feedback Intelligence — *ChicStyle*

This notebook builds and evaluates the Generative-AI feedback system using **Azure OpenAI** with
three prompting techniques — **Zero-Shot**, **Few-Shot**, and **Chain-of-Thought (CoT)** — and
compares them. It reuses the modular code in `milestone2/` (the same code the Streamlit app uses),
so there is a single implementation.

**Pipeline per review →** estimated rating (1–5), overall sentiment, per-aspect sentiment,
urgency, plus a personalized customer message and a retail-team report.

> Prerequisite: paste your key into `milestone2/.env` (`AZURE_OPENAI_API_KEY`).

## 1. Setup — import the milestone2 modules

In [4]:
import sys, os, time
# Make the milestone2 package importable and load its .env
sys.path.insert(0, os.path.abspath('milestone2'))

import pandas as pd
import numpy as np

from analyzer import analyze_review, build_context
from messaging import generate_customer_message
from reports import (generate_retail_report, analyze_dataframe,
                     summarize_by_category_urgency, urgency_pivot)
from prompts import METHOD_LABELS

print('Methods available:', METHOD_LABELS)

Methods available: {'zero_shot': 'Zero-Shot', 'few_shot': 'Few-Shot', 'cot': 'Chain-of-Thought'}


## 2. Quick sanity check — mixed sentiment & multiple aspects
The classic mixed-sentiment review from the problem statement: *"The fit is great but the color was
not as per the product image."* A good system should return **two aspects with opposite sentiments**.

In [5]:
demo = "The fit is great but the color was not as per the product image."
res = analyze_review(demo, build_context('Dresses', 'Midi Dress'), method='cot')
print('Estimated rating :', res.estimated_rating)
print('Overall sentiment:', res.overall_sentiment)
print('Urgency          :', res.urgency)
for a in res.aspects:
    print(f'  - {a.aspect}: {a.sentiment}')
print('\nReasoning:\n', res.reasoning)

Estimated rating : 4
Overall sentiment: Positive
Urgency          : Low
  - fit: Positive
  - color: Negative

Reasoning:
 Step 1: The review mentions two distinct aspects: fit and color. 
Step 2: The sentiment for 'fit' is Positive because the customer states that it is 'great'. The sentiment for 'color' is Negative because the customer indicates that it does not match the product image. 
Step 3: The overall sentiment is Positive since one aspect (fit) is praised while the other (color) is criticized. The positive aspect outweighs the negative. I estimate the product rating to be 4, as the customer is satisfied with the fit but disappointed with the color. 
Step 4: The urgency is Low because while there is a complaint about the color, it does not indicate a serious issue such as defects or dissatisfaction with the product overall.


e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...h the product overall."), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


## 3. Zero-Shot vs Few-Shot vs Chain-of-Thought
Run all three methods on the same review and inspect how they differ.

In [6]:
compare_review = "Nice material and the price was fair, but it runs a size too small and delivery took two weeks."
for key, label in METHOD_LABELS.items():
    a = analyze_review(compare_review, build_context('Bottoms', 'Jeans'), method=key)
    aspects = ', '.join(f'{x.aspect}:{x.sentiment}' for x in a.aspects)
    print(f'[{label:16}] rating={a.estimated_rating}  sentiment={a.overall_sentiment:8}  '
          f'urgency={a.urgency:6}  aspects=[{aspects}]')

e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...e immediate attention.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


[Zero-Shot       ] rating=3  sentiment=Neutral   urgency=Medium  aspects=[quality:Positive, price:Positive, sizing:Negative, delivery:Negative]


e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...='Medium', reasoning=''), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


[Few-Shot        ] rating=3  sentiment=Neutral   urgency=Medium  aspects=[material:Positive, price:Positive, sizing:Negative, delivery:Negative]
[Chain-of-Thought] rating=3  sentiment=Neutral   urgency=Medium  aspects=[material:Positive, price:Positive, sizing:Negative, delivery:Negative]


e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_... do warrant attention."), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


## 4. Method comparison on a balanced ~50-review sample
We draw a balanced sample across ratings from the Milestone-1 `cleaned_reviews.csv`, run each
method, and score predictions against the **actual** rating/sentiment (ground truth).

In [8]:
df = pd.read_csv('cleaned_reviews.csv')

# Balanced sample: 10 reviews per star rating (=50), reproducible.
# Use GroupBy.sample() directly — robust across pandas versions.
sample = (df.groupby('rating', group_keys=False)
            .sample(n=10, random_state=42)
            .reset_index(drop=True))
print('Sample size:', len(sample), '| rating counts:',
      sample['rating'].value_counts().sort_index().to_dict())

Sample size: 50 | rating counts: {1: 10, 2: 10, 3: 10, 4: 10, 5: 10}


In [9]:
def run_method(method):
    rows = []
    for _, r in sample.iterrows():
        ctx = build_context(r['department_name'], r['class_name'])
        a = analyze_review(str(r['review_text']), ctx, method=method)
        rows.append({
            'actual_rating': r['rating'],
            'actual_sentiment': r['sentiment'],
            'est_rating': a.estimated_rating,
            'pred_sentiment': a.overall_sentiment,
            'urgency': a.urgency,
            'n_aspects': len(a.aspects),
        })
    return pd.DataFrame(rows)

results = {}
timings = {}
for key, label in METHOD_LABELS.items():
    t0 = time.time()
    results[key] = run_method(key)
    timings[key] = time.time() - t0
    print(f'{label}: done in {timings[key]:.1f}s')

e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...t and design concerns.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...am to respond is high.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='par

Zero-Shot: done in 120.7s


e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...='Medium', reasoning=''), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...vel of disappointment.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='par

Few-Shot: done in 110.1s


e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...ch could affect sales.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...stomers not to buy it.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='par

Chain-of-Thought: done in 222.4s


e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...ise with a suggestion.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


In [10]:
# Build the comparison metrics table
def metrics(res, secs):
    mae = (res['est_rating'] - res['actual_rating']).abs().mean()
    exact = (res['est_rating'] == res['actual_rating']).mean() * 100
    within1 = ((res['est_rating'] - res['actual_rating']).abs() <= 1).mean() * 100
    sent_acc = (res['pred_sentiment'] == res['actual_sentiment']).mean() * 100
    return {
        'Rating MAE': round(mae, 2),
        'Rating exact %': round(exact, 1),
        'Rating within ±1 %': round(within1, 1),
        'Sentiment acc %': round(sent_acc, 1),
        'Avg #aspects': round(res['n_aspects'].mean(), 2),
        'Time (s)': round(secs, 1),
    }

comparison = pd.DataFrame({METHOD_LABELS[k]: metrics(results[k], timings[k])
                           for k in METHOD_LABELS}).T
comparison

,Rating MAE,Rating exact %,Rating within ±1 %,Sentiment acc %,Avg #aspects,Time (s)
Zero-Shot,0.54,52.0,94.0,72.0,3.64,120.7
Few-Shot,0.50,54.0,96.0,74.0,3.50,110.1
Chain-of-Thought,0.50,56.0,94.0,76.0,3.84,222.4


**Interpretation.** Lower rating MAE and higher sentiment accuracy are better. Few-Shot and
Chain-of-Thought typically separate mixed sentiment into more aspects and track the true rating
more closely than Zero-Shot — at the cost of more tokens / slightly higher latency.

## 5. Objective 3 — personalized customer messages
Thank positive customers, acknowledge neutral ones, apologize to negative ones (+ team will reach out).

In [11]:
for txt in [
    "Absolutely love this top, soft and true to size!",
    "It's okay, nothing special.",
    "The dress arrived with a broken zipper and a torn seam. Very disappointed.",
]:
    a = analyze_review(txt, method='cot')
    msg = generate_customer_message(a, txt)
    print(f'[{a.overall_sentiment}] {txt}\n  -> {msg}\n')

e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...eam to respond is low.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


[Positive] Absolutely love this top, soft and true to size!
  -> Thank you so much for your wonderful feedback! We're thrilled to hear that you love the top and that it fits you perfectly. Your satisfaction means the world to us, and we can't wait to see you in more ChicStyle pieces!



e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_... complaints or issues."), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


[Neutral] It's okay, nothing special.
  -> Thank you for taking the time to share your thoughts with us. We appreciate your feedback and are always looking for ways to enhance our offerings. If there's anything specific you'd like to see from us in the future, please let us know!



e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_... to address the issue."), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


[Negative] The dress arrived with a broken zipper and a torn seam. Very disappointed.
  -> I’m truly sorry to hear about the issues you faced with your dress. This is not the experience we want for our customers, and I assure you that a member of our team will reach out to you shortly to resolve this matter. Thank you for your understanding, and we appreciate your patience as we work to make this right.



## 6. Objective 4 — retail-team report (per review)

In [12]:
bad = "The dress arrived with a broken zipper and a torn seam. Very disappointed."
a = analyze_review(bad, build_context('Dresses', 'Evening Gown'), method='cot')
print(generate_retail_report(a, 'Dresses', 'Evening Gown', bad))

e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...s immediate attention.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


**Internal Report: Evening Gown Quality Issue**

- **Current Rating:** 1/5; overall sentiment is negative due to quality and delivery issues.
- **Customer Feedback:** Received a complaint regarding a broken zipper and torn seam upon delivery.
- **Urgency Level:** High; immediate attention required to address customer dissatisfaction.

**Recommended Action:**
- Initiate a quality control review for the evening gown production line and implement stricter inspection protocols. Additionally, reach out to the affected customer to offer a replacement or refund to restore brand trust.


## 7. Objective 2 — precompute analyzed dataset & summarize by category × urgency
We analyze a sample of the dataset with the LLM, save it to `analyzed_reviews.csv` (consumed by the
Streamlit dashboard), and summarize insights by department and urgency level.

In [13]:
# Analyze a modest sample for the dashboard (adjust `n` for cost/coverage).
n = 60
batch = df.sample(n, random_state=7).reset_index(drop=True)
analyzed = analyze_dataframe(batch, method='cot')
analyzed.to_csv('analyzed_reviews.csv', index=False)
print('Saved analyzed_reviews.csv with', len(analyzed), 'rows')
analyzed[['review_text','department_name','est_rating','pred_sentiment','urgency','n_aspects']].head()

e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...e a serious complaint."), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...ints or issues raised.'), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(
e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='par

Saved analyzed_reviews.csv with 60 rows


e:\CodeWs\GLws\capstone_project\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReviewAnalysis(estimated_...ts or dissatisfaction."), input_type=ReviewAnalysis])
  return self.__pydantic_serializer__.to_python(


,review_text,department_name,est_rating,pred_sentiment,urgency,n_aspects
0,Just received first pair in brown...love them ...,Intimate,5,Positive,Low,5
1,I saw this online and was pleased they had it ...,Tops,5,Positive,Low,3
2,I usually do not buy dresses at full price but...,Dresses,4,Positive,Low,4
3,I was lucky enough to get a hold of this intar...,Dresses,5,Positive,Low,5
4,Just got these mara hoffman classic bikini bot...,Intimate,5,Positive,Low,4


In [14]:
# Category x urgency insights (Objective 2)
print('Department x Urgency counts:')
display(urgency_pivot(analyzed))
print('\nSummary (department x urgency):')
summarize_by_category_urgency(analyzed)

Department x Urgency counts:


urgency,High,Medium,Low
department_name,,,
Dresses,2,3,15
Trend,1,0,0
Bottoms,0,1,7
Intimate,0,0,3
Jackets,0,0,3
Tops,0,3,22



Summary (department x urgency):


,department_name,urgency,reviews,avg_est_rating
0,Bottoms,Low,7,4.00
1,Bottoms,Medium,1,2.00
2,Dresses,High,2,1.00
3,Dresses,Low,15,4.20
4,Dresses,Medium,3,2.67
5,Intimate,Low,3,5.00
6,Jackets,Low,3,4.67
7,Tops,Low,22,4.32
8,Tops,Medium,3,2.33
9,Trend,High,1,2.00


## 8. Business Insights & Recommendations

Based on the Milestone-1 EDA and the Milestone-2 GenAI analysis:

1. **Prioritize by urgency, not just volume.** The system flags High-urgency reviews (defects,
   damaged/wrong items) — route these to staff immediately during festive spikes, ahead of routine
   praise.
2. **Staff response teams by negative *volume*.** Negative feedback concentrates in the highest-volume
   departments (Tops, Dresses); allocate responders proportionally.
3. **Watch mixed-sentiment aspects.** Recurring negatives on *fit/sizing* and *color vs. product image*
   indicate catalog-accuracy and size-guide fixes that reduce returns.
4. **Automate the first response.** Auto-generated apology + "a team member will reach out" messages
   give negative customers instant acknowledgement, protecting trust during emotionally important
   festive purchases.
5. **Trend department needs product review.** It shows the lowest satisfaction — investigate quality
   and supplier issues.
6. **Adopt Chain-of-Thought as the default** for production: it best separates aspects and handles
   mixed sentiment, giving the most reliable rating/urgency signal for the reports above.

**Next steps:** deploy the Streamlit app (`milestone2/app.py`), scale the precomputed analysis to a
larger sample, and feed the category/urgency summaries into the retail team's daily workflow.